In [1]:
import pandas as pd
import numpy as np
import os
import dabest
from dabest._stats_tools.confint_1group import summary_ci_1group

import warnings
warnings.simplefilter(action="ignore", category=RuntimeWarning)
warnings.simplefilter(action="ignore", category=UserWarning)
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)
warnings.simplefilter(action='ignore', category=FutureWarning)

Pre-compiling numba functions for DABEST...


Compiling numba functions: 100%|██████████| 11/11 [00:01<00:00,  9.44it/s]

Numba compilation complete!


In [2]:
laptop = "C:\\Users\\lnico"
workcomp = "C:\\Users\\User"
homecomp = "D:"
titledpath = homecomp

base_path = "\\ACC Lab Dropbox\\ACC Lab\\Nicole Lee\\eOPN3 manuscript\\Data compilation\\Trumelan\\2. Processed\\"
specifiedpath = titledpath + base_path
savedir = titledpath + "\\ACC Lab Dropbox\\ACC Lab\\Nicole Lee\\eOPN3 manuscript\\Data compilation\\Together\\Appendixstats\\Trumelan\\"

maxtime = 3600

# Responder display name mapping
responder_display_map = {
    "eOPN3": "OPN3",
    "eOPN3_ATR": "OPN3 [ATR]",
    "eOPN3-TS-ER": "OPN3-TS-ER",
    "ACR": "GtACR1",
    "ACR_ATR": "GtACR1 [ATR]",
    "PdCO": "PdCO",
}

def get_responder_display(responder):
    return responder_display_map.get(responder, responder)

In [3]:
def get_phases(light_duration_s=5.0):
    baseline = 60 * 2
    light_dur = light_duration_s * 2
    light_off = baseline + light_dur

    first_phase = slice(0, baseline - 0.5)
    light_on = slice(baseline, baseline + light_dur - 0.5)
    first_min = slice(light_off, light_off + 120 - 0.5)
    third_min = slice(light_off + 240, light_off + 360 - 0.5)
    fifth_min = slice(light_off + 480, light_off + 600 - 0.5)

    return first_phase, light_on, first_min, third_min, fifth_min

def filter_dead_flies(df):
    speed = df.iloc[:120].filter(regex="^Speed")
    alive = {c.split('_')[1] for c in speed.columns if speed[c].mean() >= 0.5}
    return df[[c for c in df.columns if c.split('_')[1] in alive]]

def load_and_split(filepath, phase_slice, split=None):
    df = pd.read_csv(filepath + "_max3600s.csv").set_index('Time (s)')
    df = filter_dead_flies(df)
    df = df.loc[phase_slice, :]
    speed_cols = sorted([c for c in df.columns if c.startswith('Speed')], key=lambda c: int(c.split('_')[1]))
    activity_cols = sorted([c for c in df.columns if c.startswith('ActivityLevel')], key=lambda c: int(c.split('_')[1]))
    if split == 'even':
        speed_cols = speed_cols[1::2]
        activity_cols = activity_cols[1::2]
    elif split == 'odd':
        speed_cols = speed_cols[0::2]
        activity_cols = activity_cols[0::2]
    return df[speed_cols].T.reset_index(drop=True), df[activity_cols].T.reset_index(drop=True)

In [4]:
def thesis_hedgesg_shared_control(df_dabest, idx_tuple, phase_group_map, metric_name, driver, responder):
    display_responder = get_responder_display(responder)
    
    db = dabest.load(df_dabest, idx=idx_tuple)
    results = db.hedges_g.results
    
    all_cols = list(idx_tuple)
    control_col = all_cols[0]
    test_cols = all_cols[1:]
    
    genotype_control = f"w1118;;UAS-{responder}/+\nw1118;{driver}/+;{driver}/+"
    genotype_test = f"w1118;{driver}/+;{driver}/{responder}"
    
    rows = []
    for col_name in all_cols:
        phase, group = phase_group_map[col_name]
        col_data = df_dabest[col_name].dropna().values
        
        group_stats = summary_ci_1group(x=col_data, func=np.mean, resamples=5000, alpha=0.05)
        
        mean_val = round(group_stats['summary'], 2)
        mean_ci_low = round(group_stats['bca_ci_low'], 2)
        mean_ci_high = round(group_stats['bca_ci_high'], 2)
        sample_size = len(col_data)
        genotype = genotype_control if group == "Control" else genotype_test
        
        if col_name in test_cols:
            res_idx = test_cols.index(col_name)
            es_val = round(results.difference.iloc[res_idx], 2)
            es_ci_low = round(results.bca_low.iloc[res_idx], 2)
            es_ci_high = round(results.bca_high.iloc[res_idx], 2)
            delta_object = "Hedges' g"
        else:
            es_val = " "
            es_ci_low = " "
            es_ci_high = " "
            delta_object = " "
        
        rows.append({
            "Driver": driver,
            "Responder": display_responder,
            "Group": group,
            "Phase": phase,
            "Genotype": genotype,
            "Sample Size": sample_size,
            "Mean": mean_val,
            "Mean_CI_low": mean_ci_low,
            "Mean_CI_high": mean_ci_high,
            "Effect Size": es_val,
            "Effect Size_CI_low": es_ci_low,
            "Effect Size_CI_high": es_ci_high,
            "Delta Object": delta_object,
            "Metric": metric_name,
        })
    
    return pd.DataFrame(rows)

## Intensity thesis stats

In [5]:
lst = ["elav", "vGAT"]
responder = "eOPN3_ATR"
w1118 = "w1118"
both_controls = "No"

first_phase, light_on, first_min, third_min, fifth_min = get_phases(5.0)

intensity_idx = ("No Light_control", "No Light_expt", "Quarter_expt", "Half_expt", "Full_expt")

intensity_phase_group = {
    'No Light_control': ('No Light', 'Control'),
    'No Light_expt': ('No Light', 'Test'),
    'Quarter_expt': ('Quarter', 'Test'),
    'Half_expt': ('Half', 'Test'),
    'Full_expt': ('Full', 'Test'),
}

for driver in lst:
    print(driver)
    
    # Build No Light control: w1118 x responder (UAS control)
    ctrl_speed, ctrl_activity = load_and_split(specifiedpath + w1118 + " x " + responder + "_0s", first_min, 'even')
    
    # Pool GAL4 control if both_controls
    if both_controls == "Yes":
        drv_speed, drv_activity = load_and_split(specifiedpath + w1118 + " x " + driver + "_ATR_0s", first_min, 'even')
        ctrl_speed = pd.concat([ctrl_speed, drv_speed], ignore_index=True)
        ctrl_activity = pd.concat([ctrl_activity, drv_activity], ignore_index=True)
    
    all_speed = [ctrl_speed.mean(axis=1)]
    all_activity = [ctrl_activity.mean(axis=1)]
    all_labels = ['No Light_control']
    
    intensity_files = {
        'No Light_expt': (specifiedpath + driver + " x " + responder + "_nolight", 'even'),
        'Quarter_expt': (specifiedpath + driver + " x " + responder + "_quarter_5s", None),
        'Half_expt': (specifiedpath + driver + " x " + responder + "_half_5s", None),
        'Full_expt': (specifiedpath + driver + " x " + responder + "_5s", 'even'),
    }
    
    for label, (filepath, split) in intensity_files.items():
        speed_df, activity_df = load_and_split(filepath, first_min, split)
        all_speed.append(speed_df.mean(axis=1))
        all_activity.append(activity_df.mean(axis=1))
        all_labels.append(label)
    
    df_dabest_speed = pd.concat(all_speed, axis=1, ignore_index=True)
    df_dabest_speed.columns = all_labels
    df_dabest_activity = pd.concat(all_activity, axis=1, ignore_index=True)
    df_dabest_activity.columns = all_labels

    df_intensity_thesis = pd.concat([
        thesis_hedgesg_shared_control(df_dabest_speed, intensity_idx, intensity_phase_group, "speed", driver, responder),
        thesis_hedgesg_shared_control(df_dabest_activity, intensity_idx, intensity_phase_group, "activitylevel", driver, responder),
    ], ignore_index=True)

    df_intensity_thesis['Assay'] = 'Intensity'
    df_intensity_thesis.to_csv(savedir + driver + " x " + responder + "_intensity_thesis_stats.csv", index=False)

print("Done!")

elav
vGAT
Done!


## Duration thesis stats

In [6]:
lst = ["elav", "vGAT"]
responder = "eOPN3_ATR"
w1118 = "w1118"
both_controls = "No"

duration_idx = ("0s_control", "0s_expt", "5s_expt", "30s_expt", "60s_expt")

duration_phase_group = {
    '0s_control': ('0s', 'Control'),
    '0s_expt': ('0s', 'Test'),
    '5s_expt': ('5s', 'Test'),
    '30s_expt': ('30s', 'Test'),
    '60s_expt': ('60s', 'Test'),
}

for driver in lst:
    print(driver)
    
    # 0s control uses get_phases(60) to align with longest duration
    _, _, first_min_0s, _, _ = get_phases(60)
    
    # Build 0s control: w1118 x responder (UAS control)
    ctrl_speed_0s, ctrl_activity_0s = load_and_split(specifiedpath + w1118 + " x " + responder + "_0s", first_min_0s, 'odd')
    
    # Pool GAL4 control if both_controls
    if both_controls == "Yes":
        drv_speed_0s, drv_activity_0s = load_and_split(specifiedpath + w1118 + " x " + driver + "_ATR_0s", first_min_0s, 'odd')
        ctrl_speed_0s = pd.concat([ctrl_speed_0s, drv_speed_0s], ignore_index=True)
        ctrl_activity_0s = pd.concat([ctrl_activity_0s, drv_activity_0s], ignore_index=True)
    
    # 0s expt
    expt_speed_0s, expt_activity_0s = load_and_split(specifiedpath + driver + " x " + responder + "_nolight", first_min_0s, 'odd')
    
    all_speed = [ctrl_speed_0s.mean(axis=1), expt_speed_0s.mean(axis=1)]
    all_activity = [ctrl_activity_0s.mean(axis=1), expt_activity_0s.mean(axis=1)]
    all_labels = ['0s_control', '0s_expt']
    
    duration_files = {
        '5s_expt': (specifiedpath + driver + " x " + responder + "_5s", 'odd'),
        '30s_expt': (specifiedpath + driver + " x " + responder + "_30s", None),
        '60s_expt': (specifiedpath + driver + " x " + responder + "_60s", None),
    }
    
    for label, (filepath, split) in duration_files.items():
        duration = int(label.split('s_')[0])
        first_phase, light_on, first_min, third_min, fifth_min = get_phases(duration)
        speed_df, activity_df = load_and_split(filepath, first_min, split)
        all_speed.append(speed_df.mean(axis=1))
        all_activity.append(activity_df.mean(axis=1))
        all_labels.append(label)
    
    df_dabest_speed = pd.concat(all_speed, axis=1, ignore_index=True)
    df_dabest_speed.columns = all_labels
    df_dabest_activity = pd.concat(all_activity, axis=1, ignore_index=True)
    df_dabest_activity.columns = all_labels

    df_duration_thesis = pd.concat([
        thesis_hedgesg_shared_control(df_dabest_speed, duration_idx, duration_phase_group, "speed", driver, responder),
        thesis_hedgesg_shared_control(df_dabest_activity, duration_idx, duration_phase_group, "activitylevel", driver, responder),
    ], ignore_index=True)

    df_duration_thesis['Assay'] = 'Duration'
    df_duration_thesis.to_csv(savedir + driver + " x " + responder + "_duration_thesis_stats.csv", index=False)

print("Done!")

elav
vGAT
Done!
